# Mean Reversion Signal (Pine → Python, for Google Colab)

Run the cells top to bottom (Shift+Enter on each). Nothing here places orders. It only **checks for a signal** and runs a quick **backtest**.

1. Settings
2. Strategy logic (your Pine script)
3. Get prices (Kraken for crypto, Yahoo for stocks)
4. Check today's signal
5. Backtest

In [ ]:
# ---------- 1. SETTINGS (same as the Pine inputs) ----------
LEN_Z        = 20     # lenZ
Z_ENTRY      = -2.0   # zEntry
RSI_LEN      = 14     # rsiLen
RSI_MAX      = 30.0   # rsiMax
USE_TREND    = True   # useTrend
TREND_LEN    = 200    # trendLen
MAX_BARS     = 10     # maxBars
STOP_PCT     = 8.0    # stopPct
EXIT_AT_MEAN = True   # exitAtMean

# Pick ONE data source:
SOURCE = "kraken"     # "kraken" (crypto) or "yahoo" (stocks)
SYMBOL = "XBTUSD"     # Kraken pair, e.g. XBTUSD, ETHUSD
# For Yahoo, use e.g. SYMBOL = "AAPL", or "SHOP.TO" for Toronto-listed stocks

In [ ]:
# ---------- 2. STRATEGY LOGIC ----------
import numpy as np
import pandas as pd

def rsi(close, length):
    """Wilder's RSI, matches ta.rsi in Pine."""
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    return 100 - 100 / (1 + avg_gain / avg_loss)

def add_signals(df):
    df = df.copy()
    df["basis"] = df["close"].rolling(LEN_Z).mean()
    df["dev"]   = df["close"].rolling(LEN_Z).std(ddof=0)   # Pine stdev = population std
    df["z"]     = np.where(df["dev"] == 0, 0.0, (df["close"] - df["basis"]) / df["dev"])
    df["rsi"]   = rsi(df["close"], RSI_LEN)
    trend_ok    = df["close"] > df["close"].rolling(TREND_LEN).mean()
    if not USE_TREND:
        trend_ok = True
    df["long_signal"] = (df["z"] <= Z_ENTRY) & (df["rsi"] <= RSI_MAX) & trend_ok
    return df

In [ ]:
# ---------- 3. GET PRICES ----------
import requests

def get_kraken_daily(pair):
    """Public endpoint, no API key needed. Returns up to ~720 daily candles."""
    r = requests.get("https://api.kraken.com/0/public/OHLC",
                     params={"pair": pair, "interval": 1440}, timeout=20)
    r.raise_for_status()
    payload = r.json()
    if payload.get("error"):
        raise RuntimeError(payload["error"])
    key = [k for k in payload["result"] if k != "last"][0]
    df = pd.DataFrame(payload["result"][key],
                      columns=["time","open","high","low","close","vwap","volume","count"])
    df["time"] = pd.to_datetime(df["time"], unit="s")
    df = df.set_index("time")[["open","high","low","close","volume"]].astype(float)
    return df.iloc[:-1]   # drop today's unfinished candle

def get_yahoo_daily(ticker):
    import yfinance as yf
    raw = yf.download(ticker, period="3y", interval="1d", auto_adjust=True, progress=False)
    raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in raw.columns]
    return raw[["open","high","low","close","volume"]].dropna()

df = get_kraken_daily(SYMBOL) if SOURCE == "kraken" else get_yahoo_daily(SYMBOL)
print(f"{len(df)} daily bars loaded for {SYMBOL}. Last bar: {df.index[-1].date()}")
df.tail()

In [ ]:
# ---------- 4. TODAY'S SIGNAL ----------
sig = add_signals(df)
last = sig.iloc[-1]
print(f"Date:        {sig.index[-1].date()}")
print(f"Close:       {last['close']:.2f}")
print(f"z-score:     {last['z']:.2f}   (needs <= {Z_ENTRY})")
print(f"RSI:         {last['rsi']:.1f}   (needs <= {RSI_MAX})")
print("BUY SIGNAL" if last["long_signal"] else "No signal")

In [ ]:
# ---------- 5. BACKTEST (long only, mirrors the Pine entry/exit rules) ----------
def backtest(sig, start_equity=25000, size_pct=0.10, commission=0.0005, slippage=0.0):
    equity, trades = start_equity, []
    pos = None
    for i in range(len(sig)):
        row = sig.iloc[i]
        if pos is not None:
            bars_held = i - pos["i"]
            stop_price = pos["entry"] * (1 - STOP_PCT / 100)
            exit_price, reason = None, None
            if row["low"] <= stop_price:
                exit_price, reason = stop_price, "Stop"
            elif bars_held >= MAX_BARS:
                exit_price, reason = row["close"], "Time exit"
            elif EXIT_AT_MEAN and row["z"] >= 0:
                exit_price, reason = row["close"], "Mean exit"
            if reason:
                ret = exit_price / pos["entry"] - 1 - 2 * commission - slippage
                pnl = pos["cash"] * ret
                equity += pnl
                trades.append({"entry_date": sig.index[pos["i"]], "exit_date": sig.index[i],
                               "entry": pos["entry"], "exit": exit_price,
                               "return_%": ret * 100, "reason": reason})
                pos = None
        if pos is None and row["long_signal"]:
            pos = {"i": i, "entry": row["close"], "cash": equity * size_pct}
    return pd.DataFrame(trades), equity

trades, final_equity = backtest(sig)
if trades.empty:
    print("No trades in this period. The signal is strict, so that can be normal.")
else:
    print(f"Trades: {len(trades)}   Win rate: {(trades['return_%'] > 0).mean():.0%}   "
          f"Avg return: {trades['return_%'].mean():.2f}%")
    print(f"Start $25,000 -> End ${final_equity:,.0f}")
trades